# Arbeiten mit Dateien

Fast immer kommt der Input für ein Programm aus Dateien.

Python unterstützt die Arbeit mit Dateien und Ordnern mit einer Reihe von Modulen in der Standard Library, darunter [`os`](https://docs.python.org/3/library/os.html), [`shutil`](https://docs.python.org/3/library/shutil.html), [`tempfile`](https://docs.python.org/3/library/tempfile.html), und [`pathlib`](/https://docs.python.org/3/library/pathlib.html).  

Die Funktionalität überschneidet sich dabei teilweise. Hier folgen Empfehlungen für einige übliche Operationen.

## Aktuelles Verzeichnis sehen und ändern

In [ ]:
import os

start_dir = os.getcwd()
start_dir

In [ ]:
os.chdir('..')
os.getcwd()

**Übung**: Was passiert, wenn man die Zelle oben mehrmals ausführt?

## Dateipfade 

Das `os` Modul speichert Pfade als Strings. Das ist fehleranfällig und etwas umständlich.

Als Lösung wurde mit Python 3.4 das `pathlib` Modul eingeführt. Es bietet ein `Path` Objekt, um Pfade zu repräsentieren. 


In [ ]:
from pathlib import Path

Path(os.getcwd())


`Path` Objekte passen sich automatisch an das Betriebssystem an und bieten nützliche Funktionalität.

In [ ]:
readme = Path('README.md').resolve() # resolve löst den ganzen Pfad auf
readme

In [ ]:
# Beinhaltendes Verzeichnis
readme.parent

In [ ]:
# Dateiname
readme.name

In [ ]:
# Dateiendung
readme.suffix

In [ ]:
# Größe des Files in Bytes
readme.stat().st_size

## Inhalte im Verzeichnis auflisten

In [ ]:
notebooks = Path('02-data').resolve() 


list(notebooks.iterdir())


## Dateien nach Filenamen finden


In [ ]:
# Finde alle Notebooks im Verzeichnis, die mit 0 beginnen.
list(notebooks.glob('0*.ipynb'))

## Verzeichnisse anlegen

Neue Pfade können mit `pathlib` über den `/` Operator konstruiert werden. Wenn der Code auf Windows läuft, wird trotzdem ein Backslash erzeugt.


In [ ]:
# Pfad zu Unterverzeichnis des Arbeitsverzeichnisses, das noch nicht existiert
subdir = Path() / 'unterordner'
# Verzeichnis anlegen
subdir.mkdir(exist_ok=True)


## Dateien kopieren und verschieben

Das `shutil` Modul (Shell Utilities) bietet Funktionen für das Kopieren, Verschieben, und Löschen von Daten.

In [ ]:
import shutil

old_path = Path('README.md')
new_path = subdir / 'README2.md'

# File kopieren
shutil.copy(old_path, new_path)

# Analog File verschieben
# shutil.move(old_path, new_path)


Achtung: Wenn das Zielfile schon existiert, wird es dabei überschrieben!

## Textdateien lesen und schreiben

Auch um Textdateien zu lesen und zu schreiben, ist `pathlib` die einfachste Option. 

In [ ]:
# String in neues File schreiben
new_file = Path('.') / 'hello.txt'
new_file.write_text('Hello World!')

In [ ]:
# README.md als String einlesen
readme.read_text()

Das sieht nicht ganz richtig aus - Umlaute werden falsch angezeigt.


### Encodings

Alle Files bestehen aus Bytes. Um Text zu speichern, muss er also in Bytes verwandelt werden. Dafür gibt es (leider) verschiedene Möglichkeiten - sogenannte *Encodings*.

Historisch war ASCII mit 8 Bits (1 Byte) pro Zeichen lange der Standard, damit lässt sich aber nur englischer Text verlustfrei codieren - Umlaute sind z.B. nicht vorgesehen.

Mittlerweile hat sich UTF-8 als Standard etabliert. Mit einer variablen Anzahl Bytes kann Unicode, also Zeichen für alle Sprachen der Welt (inklusive Emojis 👌), kodiert werden.

Wenn wir wissen, dass unser File in UTF-8 kodiert wurde, können wir das angeben:


In [ ]:
readme.read_text(encoding='utf-8')

Immer noch recht unübersichtlich - der gesamte Text wird in einem langen String angezeigt.



Das `\n` zeigt einen Zeilenumbruch an - daran lässt sich der Text aufteilen:

In [ ]:
text = readme.read_text(encoding='utf-8')
lines = text.splitlines()
lines[0]

## Was ist mit CSV-Dateien?

Da CSV-Dateien auch Textdateien sind, ist es einfach sie so einzulesen.

### Übung

Lese die Daten im CSV-File data/simple.csv ein und geben sie mit `print` aus.

<details>

<summary>Tipps</summary>

- Zeilenweise arbeiten
- `.split(',')` nutzen, um Zeilen aufzuteilen
</details>

Bonus: Speichere die Zeit- und die Datenwerte jeweils in eine Liste.


In [ ]:
os.chdir(start_dir)  # zurück zum Anfang
csv_path = Path('data/simple.csv')

In der Praxis ist dieser Ansatz eher mühsam und fehleranfällig. Das `pandas` Paket bietet dafür eine elegante fertige Lösung.

`pandas` unterstützt außerdem auch viele binäre Formate, insbesondere Excelfiles.

# Files öffnen und lesen

`path.read_text()` öffnet ein File, liest es komplett, und schließt es wieder.

Manchmal wollen wir mehr Kontrolle: z.B. das File nur zeilenweise bearbeiten oder gemischt lesen und schreiben.

In [ ]:
with open(csv_path, mode="rt") as f:  # 'with open' ist ein Context Manager
    for line in f:
        try:
            val = float(line.strip().split(',')[1])
        except ValueError:
            continue
        if val > 2:
            print(line)
            # Rest des Files wird nie gelesen.
            break
        
# Context Manager schließt das File automatisch (auch bei Exceptions).


## CSV-File zeilenweise schreiben

In [ ]:
import random 

# Daten erzeugen...

# List comprehensions - Kurzform von for-loops
values_1 = [random.randint(0, 10) for _i in range(20)]
values_2 = [v % 2 == 0 for v in values_1]

In [ ]:
# Äquivalent zu:
values_1 = []
for _i in range(20):
    values_1.append(random.randint(0,10))

values_2 = []
for v in values_1:
    values_2.append(v % 2 == 0)

In [ ]:
# ...und schreiben 
with open(Path('data/rand.csv'), 'wt') as f:
    f.write('val, is_even\n')
    # Iteriere parallel über beide Listen:
    for v1, v2 in zip(values_1, values_2):
        f.write(f'{v1}, {v2}\n')